<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Individual-stock-returns-from-CRSP-for-all-firms-listed-in-the-NYSE,-AMEX,-and-NASDAQ" data-toc-modified-id="Individual-stock-returns-from-CRSP-for-all-firms-listed-in-the-NYSE,-AMEX,-and-NASDAQ-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Individual stock returns from CRSP for all firms listed in the NYSE, AMEX, and NASDAQ</a></span></li><li><span><a href="#stock-level-predictive-characteristics" data-toc-modified-id="stock-level-predictive-characteristics-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>stock-level predictive characteristics</a></span><ul class="toc-item"><li><span><a href="#Stock-Characteristics-Processing" data-toc-modified-id="Stock-Characteristics-Processing-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>Stock Characteristics Processing</a></span></li></ul></li><li><span><a href="#Eight-macroeconomic-predictors" data-toc-modified-id="Eight-macroeconomic-predictors-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Eight macroeconomic predictors</a></span></li><li><span><a href="#Merge-macroeconomic-predictors-and-Stock-Characteristics" data-toc-modified-id="Merge-macroeconomic-predictors-and-Stock-Characteristics-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Merge macroeconomic predictors and Stock Characteristics</a></span></li></ul></div>

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import rankdata

### Individual stock returns from CRSP for all firms listed in the NYSE, AMEX, and NASDAQ

In [ ]:
stock_returns = pd.read_csv('data/raw/kjtt6tbmn0gz9chy.csv')

In [ ]:
stock_returns.head()

In [ ]:
# Returns data downloaded from CRSP contains “C” and “B”, first replace them with nan.
stock_returns['RET'].replace(['B', 'C'], [np.nan, np.nan], inplace=True)
stock_returns['RET'] = stock_returns['RET'].astype(float)

### stock-level predictive characteristics
Include 94 characteristics and industry dummies

Download: https://dachxiu.chicagobooth.edu/

In [ ]:
stock_characteristics = pd.read_csv('data/raw/datashare.csv')

In [ ]:
stock_characteristics.head()

#### Stock Characteristics Processing
1、Merge characteristics and returns data

2、We cross-sectionally rank all stock characteristics period-by-period and map these ranks into the [-1,1]

3、Another issue is missing characteristics, which we replace with the cross-sectional median at each month for each stock, respectively.

In [ ]:
# merge data
characteristics_with_return = pd.merge(stock_characteristics, stock_returns, \
                                       left_on=['permno', 'DATE'], right_on=['PERMNO', 'date'], how='left')

In [ ]:
# Delete the row where the return is nan
characteristics_with_return.dropna(subset=['RET'], inplace=True)
characteristics_with_return = characteristics_with_return.drop(columns=['PERMNO', 'date'])

Here is an example that shows that the null value is not involved in sorting and maps the feature to [-1, 1].

In [ ]:
test = np.array([[1,9,3], [4, 1, 6], [7, np.nan, 9]])
test

In [ ]:
test = rankdata(test, axis=0, nan_policy='omit') 
test

In [ ]:
test = ((test-1) / (np.nanmax(test, axis=0) - 1) - 0.5) * 2
test

In [ ]:
def Normal(x):
    
    x = rankdata(x, axis=0, nan_policy='omit')
    
    x = ((x-1) / (np.nanmax(x, axis=0) - 1) - 0.5) * 2
    
    return x

In [ ]:
# cross-sectionally rank
feature_set = stock_characteristics.columns[2:-1]
characteristics_with_return[feature_set] = characteristics_with_return.groupby('DATE')[feature_set].transform(lambda x: Normal(x.values))

In [ ]:
characteristics_with_return.head()

In [ ]:
### number of sic
print("number of sic:{}".format(len(characteristics_with_return.sic2.unique())))

In [ ]:
# 使用OneHotEncoder对行业代码进行one-hot编码
industry_encoded = pd.get_dummies(characteristics_with_return['sic2'], prefix='sic2')

In [ ]:
industry_encoded.iloc[:10]

In [ ]:
characteristics_with_return = pd.concat([characteristics_with_return, industry_encoded], axis=1)

In [ ]:
characteristics_with_return.head()

### Eight macroeconomic predictors

including dividend-price ratio (dp), earnings-price ratio (ep), book-to-market ratio (bm), net equity expansion (ntis), Treasury-bill rate (tbl), term spread (tms), default spread (dfy), and stock variance (svar)

Download https://docs.google.com/spreadsheets/d/1bM7vCWd3WOt95Sf9qjLPZjoiafgF_8EG/edit#gid=407859737

In [ ]:
macroeconomic_predictors = pd.read_csv('data/raw/PredictorData2023.xlsx - Monthly.csv', thousands=",")

In [ ]:
macroeconomic_predictors.head()

dp: The Dividend Price Ratio (d/p) is the difference between the log of dividends and the log of prices.

ep: The Earnings Price Ratio (e/p) is the difference between the log of earnings and the log of prices.

tms: The Term Spread (tms) is the difference between the long term yield on government bonds and the Treasury-bill.

dfy: The Default Yield Spread (dfy) is the difference between BAA and AAA-rated corporate bond yields.

In [ ]:
macroeconomic_predictors['dp'] = np.log(macroeconomic_predictors['D12']) - np.log(macroeconomic_predictors['Index'])
macroeconomic_predictors['EP'] = np.log(macroeconomic_predictors['E12']) - np.log(macroeconomic_predictors['Index'])
macroeconomic_predictors['tms'] = macroeconomic_predictors['lty'] - macroeconomic_predictors['tbl']
macroeconomic_predictors['dfy'] = macroeconomic_predictors['BAA'] - macroeconomic_predictors['AAA']

In [ ]:
macroeconomic_predictors = macroeconomic_predictors[['yyyymm', 'dp', 'EP', 'b/m', 'ntis', 'tbl', 'tms', 'dfy', 'svar']]

In [ ]:
macroeconomic_predictors.head()

### Merge macroeconomic predictors and Stock Characteristics

In [ ]:
characteristics_with_return['DATE'] = characteristics_with_return['DATE'].astype(str).str[:6].astype(int)

data = pd.merge(characteristics_with_return, macroeconomic_predictors, left_on=['DATE'], right_on=['yyyymm'], how='left')

In [ ]:
data.head()

In [ ]:
# 计算超额收益
data['RET'] = data['RET'] - data['tbl']

In [ ]:
macroeconomic_columns = ['dp', 'EP', 'b/m', 'ntis', 'tbl', 'tms', 'dfy', 'svar']
len(macroeconomic_columns)

In [ ]:
len(feature_set)

In [ ]:
interaction1_array = data[feature_set].values
interaction2_array = data[macroeconomic_columns].values

In [ ]:
# 计算交互项
interaction_array = interaction1_array[:, :, np.newaxis] * interaction2_array[:, np.newaxis, :]

In [ ]:
# 生成新列名
interaction_columns = [f'{f1}_{f2}' for f1 in feature_set for f2 in macroeconomic_columns]

In [ ]:
len(interaction_columns)

In [ ]:
# 将交互项展平并转换为 DataFrame
interaction_df = pd.DataFrame(interaction_array.reshape(data.shape[0], -1), columns=interaction_columns)

In [ ]:
interaction_df.head()

In [ ]:
# 将交互项加入原 DataFrame
data = pd.concat([data, interaction_df], axis=1)

In [ ]:
data.head()

In [ ]:
data.drop(columns=['yyyymm', 'sic2'] + macroeconomic_columns, inplace=True)

In [ ]:
data.head()

In [ ]:
import pickle
with open('data.pkl', 'wb') as handle:
    pickle.dump(data, handle)